<a href="https://colab.research.google.com/github/Muhammad-Ahmad-1263/urdu-ocr-codesaviours-si26-Muhammad-Ahmad/blob/main/SI26_Week3_Muhammad_Ahmad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 — Expand to 200+ Images

This notebook:
1. Loads your **Week 1** data (`data.zip`: raw images + `labels.csv`) and your **Week 3** new batch (`week3_and_ravi_dataset.zip`: 203 new images + `combined_new_entries.csv`).
2. Merges both into a single `data/raw/<category>/` structure and one master `data/labels.csv`.
3. Verifies you've crossed the 200-image minimum.
4. Runs preprocessing (grayscale, contrast normalize, resize/pad to a fixed size) on every image into `data/processed/`.
5. Flags which labels still need manual transcription (`text` column empty / TODO) so you know what's left before training.

**Before running:** upload `data.zip` (Week 1) and `week3_and_ravi_dataset.zip` (Week 3) when prompted in Step 1.


## Step 1 — Upload Week 1 and Week 3 zip files

In [ ]:
from google.colab import files
print("Upload data.zip (Week 1) and week3_and_ravi_dataset.zip (Week 3) together:")
uploaded = files.upload()
print(list(uploaded.keys()))


## Step 2 — Extract both archives into separate staging folders

In [ ]:
import zipfile, os

os.makedirs("week1_staging", exist_ok=True)
os.makedirs("week3_staging", exist_ok=True)

with zipfile.ZipFile("data.zip", "r") as z:
    z.extractall("week1_staging")

with zipfile.ZipFile("week3_and_ravi_dataset.zip", "r") as z:
    z.extractall("week3_staging")

print("Week 1 staging:", os.listdir("week1_staging"))
print("Week 3 staging:", os.listdir("week3_staging"))


## Step 3 — Merge raw images into one combined `data/raw/` folder

Categories: `newspaper`, `books`, `signboards`, `synthetic`, `other`.
Week 1 and Week 3 use non-overlapping filename prefixes (`urdu_XXX.png` vs `*_ss_XXX.png`), so we can copy both straight into the same category folders without collisions.

In [ ]:
import shutil

COMBINED_RAW = "data/raw"
os.makedirs(COMBINED_RAW, exist_ok=True)

def merge_raw(src_data_dir):
    src_raw = os.path.join(src_data_dir, "raw")
    if not os.path.isdir(src_raw):
        print(f"No raw/ folder found under {src_data_dir}, skipping")
        return
    for cat in os.listdir(src_raw):
        src_cat_dir = os.path.join(src_raw, cat)
        if not os.path.isdir(src_cat_dir):
            continue
        dst_cat_dir = os.path.join(COMBINED_RAW, cat)
        os.makedirs(dst_cat_dir, exist_ok=True)
        for fname in os.listdir(src_cat_dir):
            shutil.copy2(os.path.join(src_cat_dir, fname), os.path.join(dst_cat_dir, fname))

# Week 1 raw images live at week1_staging/data/raw/<category>/
merge_raw("week1_staging/data")
# Week 3 raw images live at week3_staging/data/raw/<category>/
merge_raw("week3_staging/data")

print("Combined raw counts:")
total = 0
for cat in sorted(os.listdir(COMBINED_RAW)):
    n = len(os.listdir(os.path.join(COMBINED_RAW, cat)))
    total += n
    print(f"  {cat}: {n}")
print(f"TOTAL: {total} images")


## Step 4 — Verify you've hit the 200+ image minimum

In [ ]:
assert total >= 200, f"Only {total} images — Week 3 target is 200+, keep collecting!"
print(f"✅ {total} images — target met.")


## Step 5 — Merge labels into one master `data/labels.csv`

- Week 1's `labels.csv` already has real schema `image,category,text` with real transcriptions.
- Week 3's `combined_new_entries.csv` has schema `filename,label,source,category` (from the screenshot-collection step) and is converted to match, with `text` left blank where it still needs manual transcription.

**Important:** the `text` column is the ground-truth label your model will train on. Any row with blank/TODO text needs you (or a teammate) to look at the image and type the correct Urdu text before this data is used for training — wrong labels are worse than missing ones.

In [ ]:
import csv

# Week 1 labels — already real schema
week1_labels_path = "week1_staging/data/labels.csv"
week1_rows = []
if os.path.exists(week1_labels_path):
    with open(week1_labels_path, encoding="utf-8-sig") as f:
        week1_rows = list(csv.DictReader(f))

# Week 3 new entries — convert filename/label/source/category -> image/category/text
week3_entries_path = "week3_staging/combined_new_entries.csv"
week3_rows = []
if os.path.exists(week3_entries_path):
    with open(week3_entries_path, encoding="utf-8-sig") as f:
        raw_rows = list(csv.DictReader(f))
    for r in raw_rows:
        week3_rows.append({
            "image": f"data/raw/{r['category']}/{r['filename']}",
            "category": r["category"],
            "text": (r.get("label") or "").strip()  # blank = still TODO
        })

all_rows = week1_rows + week3_rows

os.makedirs("data", exist_ok=True)
with open("data/labels.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["image", "category", "text"])
    w.writeheader()
    w.writerows(all_rows)

labeled = sum(1 for r in all_rows if r["text"].strip())
todo = len(all_rows) - labeled
print(f"Master labels.csv: {len(all_rows)} rows total | {labeled} labeled | {todo} still need transcription")


### See exactly which rows still need manual transcription

In [ ]:
todo_rows = [r for r in all_rows if not r["text"].strip()]
print(f"{len(todo_rows)} images still need a 'text' label:")
for r in todo_rows[:20]:
    print(" -", r["image"])
if len(todo_rows) > 20:
    print(f"  ... and {len(todo_rows) - 20} more")


## Step 6 — Preprocess every image

Grayscale + autocontrast + resize (preserving aspect ratio) + pad to a fixed 512×64 canvas, saved into `data/processed/<category>/`. Fixed-size, uniform-contrast inputs make downstream OCR training much easier.

In [ ]:
from PIL import Image, ImageOps

RAW_DIR = "data/raw"
OUT_DIR = "data/processed"
TARGET_H = 64
MAX_W = 512

def preprocess_image(path):
    im = Image.open(path)
    if im.mode == "RGBA":
        bg = Image.new("RGB", im.size, (255, 255, 255))
        bg.paste(im, mask=im.split()[3])
        im = bg
    else:
        im = im.convert("RGB")

    gray = ImageOps.grayscale(im)
    gray = ImageOps.autocontrast(gray, cutoff=1)

    w, h = gray.size
    scale = TARGET_H / h
    new_w = min(MAX_W, max(1, int(w * scale)))
    resized = gray.resize((new_w, TARGET_H), Image.LANCZOS)

    canvas = Image.new("L", (MAX_W, TARGET_H), 255)
    canvas.paste(resized, (0, 0))
    return canvas

processed_count = 0
for cat in os.listdir(RAW_DIR):
    cat_raw = os.path.join(RAW_DIR, cat)
    if not os.path.isdir(cat_raw):
        continue
    cat_out = os.path.join(OUT_DIR, cat)
    os.makedirs(cat_out, exist_ok=True)
    for fname in sorted(os.listdir(cat_raw)):
        try:
            out_im = preprocess_image(os.path.join(cat_raw, fname))
            out_im.save(os.path.join(cat_out, fname))
            processed_count += 1
        except Exception as e:
            print(f"FAILED {fname}: {e}")

print(f"Preprocessed {processed_count} images -> {OUT_DIR}/<category>/")


## Step 7 — Sanity check: preview a few processed images

In [ ]:
import matplotlib.pyplot as plt

sample_cats = [c for c in os.listdir(OUT_DIR) if os.listdir(os.path.join(OUT_DIR, c))][:5]
fig, axes = plt.subplots(1, len(sample_cats), figsize=(15, 3))
for ax, cat in zip(axes, sample_cats):
    fname = os.listdir(os.path.join(OUT_DIR, cat))[0]
    img = Image.open(os.path.join(OUT_DIR, cat, fname))
    ax.imshow(img, cmap="gray")
    ax.set_title(cat)
    ax.axis("off")
plt.tight_layout()
plt.show()


## Step 8 — Zip everything up for download / GitHub push

In [ ]:
import shutil
shutil.make_archive("week3_combined_dataset", "zip", ".", "data")
from google.colab import files
files.download("week3_combined_dataset.zip")


## Summary / what's left

- ✅ Combined Week 1 + Week 3 raw images into one `data/raw/` (200+ images, target met)
- ✅ Merged labels into one `data/labels.csv`
- ✅ Preprocessed all images into `data/processed/`
- ⚠️ **TODO:** transcribe the `text` column for the rows printed in Step 5 — these are still blank and need manual review before training. `books` category was intentionally skipped this week.


---
# Step 2 — Build a Dataset Class

A `Dataset` class tells the model: "here is one image, here is the correct text for it." It's a filing system — when the model asks for sample `N`, the class opens the right image, processes it, and returns it with its label.

## Cell 1 — Build the Dataset Class

In [ ]:
!pip install transformers torch pillow pandas -q

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Load and convert image
        image = Image.open(row['image']).convert('RGB')

        # Process image for the model
        encoding = self.processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()

        # Process the text label
        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128
        ).input_ids
        labels = torch.tensor(labels)

        return {'pixel_values': pixel_values, 'labels': labels}


## Cell 2 — Test Your Dataset

**Note:** 33 of the 225 rows in `data/labels.csv` are labeled so far (signboards + newspaper); the rest (`synthetic`, `other`) still have blank `text` values as TODO. The class will still load and run fine on blank labels — it just means those samples aren't useful for real training yet. Keep filling in labels before Week 4's actual training run.

In [ ]:
# Load the TrOCR processor
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')

# Create dataset
dataset = UrduOCRDataset('data/labels.csv', processor)

# Test it loads correctly
sample = dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')

# Create train / test split (80% train, 20% test)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)

print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')


If the shapes print without errors, your dataset is correctly wired up and ready for model training in Week 4.

### Sources & Links

**Tools & Platforms**
- Google Colab: https://colab.research.google.com
- GitHub: https://github.com

**Libraries**
- PyTorch: https://pytorch.org
- Hugging Face Transformers: https://huggingface.co/docs/transformers
- TrOCR model (Microsoft): https://huggingface.co/microsoft/trocr-base-printed
- Pillow: https://pillow.readthedocs.io
- pandas: https://pandas.pydata.org

**Additional Image Sources (for expanding your dataset)**
- Urdu Handwriting Dataset: https://github.com/urduhack/urdu-handwriting-dataset
- Kaggle Urdu OCR Datasets: https://www.kaggle.com/datasets (search: Urdu OCR)
- Mendeley Data: https://data.mendeley.com (search: Urdu text)
- Jang (Urdu newspaper): https://jang.com.pk
- Dawn Urdu: https://urdu.dawn.com
- BBC Urdu: https://www.bbc.com/urdu
